In [ ]:
from pystac.client import Client
from odc.stac import load
from odc.geo.geom import BoundingBox
from rasterio import Env
import folium
import matplotlib.pyplot as plt

from src.utils import process_mangroves, get_gmw

In [ ]:
# STAC APIs and Collection names
csde_catalog = "https://stac.dataspace.copernicus.eu/v1"
s2_cloudless_mosaic = "sentinel-2-global-mosaics"
dep_catalog = "https://stac.digitalearthpacific.org/"
s2_geomad = "dep_s2_geomad"

# Bounding box in PNG, in the mangrove areas near Kikori
bbox = BoundingBox(left=144.35, bottom=-7.75, right=144.48, top=-7.65)
bbox.explore()

In [ ]:
cdse = Client.open(csde_catalog)
dep = Client.open(dep_catalog)

In [ ]:
cdse_items = cdse.search(
    collections=[s2_cloudless_mosaic], bbox=bbox, datetime="2025"
).item_collection()

print(f"Found {len(cdse_items)} items in the CSDE catalog for the bounding box.")

In [ ]:
data = load(
    cdse_items,
    bbox=bbox,
    group_by="solar_day",
    chunks={"x": 1024, "y": 1024},
    measurements=["B08", "B04", "B03", "B02"],
)

# Rename to RGBNIR for consistency with the DEP GeoMAD dataset
data = data.rename({"B08": "nir", "B04": "red", "B03": "green", "B02": "blue"})

data

In [ ]:
with Env(
    profile_name="copernicus",
    AWS_S3_ENDPOINT="eodata.dataspace.copernicus.eu",
    AWS_VIRTUAL_HOSTING="FALSE",
):
    cdse_loaded = data.squeeze().compute()

cdse_loaded

In [ ]:
cdse_loaded[["red", "green", "blue"]].to_array().plot.imshow(
    col="time", col_wrap=2, vmin=0, vmax=2500, rgb="variable", size=6
)

In [ ]:
dep_items = dep.search(
    collections=[s2_geomad], bbox=bbox, datetime="2025"
).item_collection()

print(f"Found {len(dep_items)} items in the DEP catalog for the bounding box.")

In [ ]:
dep_loaded = load(
    dep_items,
    bbox=bbox,
    measurements=["nir", "red", "green", "blue"],
    chunks={"x": 1024, "y": 1024},
).squeeze().compute()

In [ ]:
dep_loaded[["red", "green", "blue"]].to_array().plot.imshow(
    col_wrap=2, vmin=0, vmax=2500)

In [ ]:
m = folium.Map(location=[-7.7, 144.4], zoom_start=12)

cdse_loaded.isel(time=0).odc.explore(bands=["red", "green", "blue"], name="CSDE Sentinel-2 Mosaic", map=m, vmin=0, vmax=2500)
dep_loaded.odc.explore(bands=["red", "green", "blue"], name="DEP GeoMAD", map=m, vmin=0, vmax=2500)

folium.LayerControl().add_to(m)
m

In [ ]:
areas = get_gmw()

cdse_mangroves = process_mangroves(cdse_loaded.isel(time=0), areas)
dep_mangroves = process_mangroves(dep_loaded, areas)

In [ ]:
m2 = folium.Map(location=[-7.7, 144.4], zoom_start=12)
cdse_mangroves.mangroves.odc.explore(cmap="Greens", name="CSDE Mangroves", map=m2, vmin=0, vmax=2)
dep_mangroves.mangroves.odc.explore(cmap="Greens", name="DEP Mangroves", map=m2, vmin=0, vmax=2)

folium.LayerControl().add_to(m2)
m2

In [ ]:
# Do a side-by-side RGB plot of the two datasets to visually compare them
vmin, vmax = 0, 2000

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cdse_loaded.isel(time=0)[["red", "green", "blue"]].to_array().plot.imshow(
    rgb="variable", ax=axes[0], add_colorbar=False, vmin=vmin, vmax=vmax
)
axes[0].set_title("CDSE Sentinel-2 Mosaic")
axes[0].axis("off")

dep_loaded[["red", "green", "blue"]].to_array().plot.imshow(
    rgb="variable", ax=axes[1], add_colorbar=False, vmin=vmin, vmax=vmax
)
axes[1].set_title("DEP GeoMAD")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

vmin, vmax = 0, 2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cdse_mangroves.where(cdse_mangroves.mangroves != 255).mangroves.plot.imshow(
    ax=axes[0], cmap="Greens", add_colorbar=False, vmin=vmin, vmax=vmax
)
axes[0].set_title("CDSE Mangroves")
axes[0].axis("off")

dep_mangroves.where(dep_mangroves.mangroves != 255).mangroves.plot.imshow(
    ax=axes[1], cmap="Greens", add_colorbar=False, vmin=vmin, vmax=vmax
)
axes[1].set_title("DEP Mangroves")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Compare mangroves across the first four original CDSE times
vmin, vmax = 0, 2

times_to_plot = min(4, data.sizes.get("time", 0))
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

# Save processed mangroves so the next cell can reuse them (no reloading)
cdse_mangroves_4 = []

for i in range(times_to_plot):
    cdse_t = cdse_loaded.isel(time=i).squeeze()

    mang_t = process_mangroves(cdse_t, areas)
    mang_arr = mang_t.where(mang_t.mangroves != 255).mangroves
    cdse_mangroves_4.append(mang_arr)

    mang_arr.plot.imshow(
        ax=axes[i], cmap="Greens", add_colorbar=False, vmin=vmin, vmax=vmax
    )
    time_label = str(data.time.values[i])[:10]
    axes[i].set_title(f"CDSE Mangroves - {time_label}")
    axes[i].axis("off")

for j in range(times_to_plot, 4):
    axes[j].axis("off")

plt.tight_layout()
plt.show()